# 🔀 Notebook 5 — Merge Adapters (TIES)
**Fully self-contained — nothing to upload.**

TIES-merges the trained tier adapters into one final unified adapter.

**Prerequisite:** Notebooks 2, 3, 4 must be done (or at least 2 of 3).  
**Output:** `My Drive/hoichoi/adapters/merged/`  
**Merge weights:** `high=0.5, mid=0.3, base=0.2` (biased toward switch-dense data)

---
⏭️ **Next:** Open `Notebook_6_Evaluate.ipynb`

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CELL 1 — Mount Drive & Check Adapter Readiness
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
import os, subprocess, sys

# 1. Mount Google Drive (if in Colab)
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_ROOT = '/content/drive/MyDrive/hoichoi'
except Exception:
    DRIVE_ROOT = '/content/hoichoi'

ADAPTER_ROOT = DRIVE_ROOT + '/adapters'
os.makedirs(ADAPTER_ROOT, exist_ok=True)

# 2. Google Drive links for your trained adapters
GDRIVE_LINKS = {
    'high': 'https://drive.google.com/drive/folders/1Se4DsbBA4M6jtjV7PiAwZTlDYBrWZvoT?usp=sharing',
    'base': 'https://drive.google.com/drive/folders/1ZgRmgrZV-kYx6JfsS3Rk2eRcbFaJFNNX?usp=sharing',
    'mid':  '',  # Add mid link here if trained, or leave empty
}

def is_ready(p):
    cfg_ok = os.path.exists(f'{p}/adapter_config.json')
    bin_ok = os.path.exists(f'{p}/adapter_model.safetensors') or os.path.exists(f'{p}/adapter_model.bin')
    return cfg_ok and bin_ok

print('Checking adapter readiness...')
ready_adapters = {}

for tier in ['high', 'mid', 'base']:
    p = f'{ADAPTER_ROOT}/{tier}/best'
    
    # If not on Drive yet, auto-download from shared Google Drive link
    if not is_ready(p) and tier in GDRIVE_LINKS and GDRIVE_LINKS[tier].strip():
        print(f'  ⬇️  Downloading {tier} tier from shared Google Drive link...')
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'gdown'], capture_output=True)
        os.makedirs(p, exist_ok=True)
        subprocess.run(['gdown', '--folder', GDRIVE_LINKS[tier], '-O', p], capture_output=True)
        
    if is_ready(p):
        print(f'  {tier:6s} : ✅ Ready ({p})')
        ready_adapters[tier] = p
    else:
        print(f'  {tier:6s} : ❌ Not found ({p})')

if len(ready_adapters) < 2:
    raise RuntimeError(f'❌ Need at least 2 trained adapters to merge. Found: {list(ready_adapters.keys())}')
elif len(ready_adapters) < 3:
    print(f'\n⚠️ Merging with {len(ready_adapters)} ready adapters: {list(ready_adapters.keys())} (high + base).')
else:
    print(f'\n✅ All 3 adapters ready for TIES merge: {list(ready_adapters.keys())}!')

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CELL 2 — Install Dependencies & Check GPU
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'transformers>=4.40.0', 'peft>=0.10.0', 'accelerate>=0.29.0'], capture_output=True)
print('✅ Dependencies installed.')

import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Running on device: {device}')

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CELL 3 — TIES Merging (Inline, Self-Contained)
# Trims bottom 30% conflicting parameter directions (density=0.7)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
import os, json, torch
from pathlib import Path
from transformers import WhisperForConditionalGeneration, WhisperProcessor
from peft import PeftModel

BASE_MODEL_ID = 'SayedShaun/bengali-whisper-medium'
MERGED_DIR    = DRIVE_ROOT + '/adapters/merged'
os.makedirs(MERGED_DIR, exist_ok=True)

# 1. Calculate normalized weights (high=0.5, mid=0.3, base=0.2)
raw_weights = {'high': 0.5, 'mid': 0.3, 'base': 0.2}
filtered_weights = {t: raw_weights[t] for t in ready_adapters}
weight_sum = sum(filtered_weights.values())
norm_weights = {t: w / weight_sum for t, w in filtered_weights.items()}
print(f'Merge weights: {norm_weights}')

# 2. Load Base Model
print(f'Loading base model: {BASE_MODEL_ID} ...')
base_model = WhisperForConditionalGeneration.from_pretrained(
    BASE_MODEL_ID, torch_dtype=torch.float16 if device == 'cuda' else torch.float32
)
base_model.config.use_cache = False

# 3. Load Adapters into PEFT Model
tiers_ordered = [t for t in ['high', 'mid', 'base'] if t in ready_adapters]
first_tier = tiers_ordered[0]
print(f'Attaching initial adapter: {first_tier} ...')
peft_model = PeftModel.from_pretrained(base_model, ready_adapters[first_tier], adapter_name=first_tier)

for t in tiers_ordered[1:]:
    print(f'Loading secondary adapter: {t} ...')
    peft_model.load_adapter(ready_adapters[t], adapter_name=t)

# 4. Perform TIES Merge
DENSITY = 0.7  # keeps top 70% magnitude deltas, trims conflicting 30%
print(f'Executing TIES merge (density={DENSITY}) ...')
peft_model.add_weighted_adapter(
    adapters         = tiers_ordered,
    weights          = [norm_weights[t] for t in tiers_ordered],
    adapter_name     = 'merged',
    combination_type = 'ties',
    density          = DENSITY,
)
peft_model.set_adapter('merged')

# 5. Save Merged Adapter & Processor
print('Saving merged adapter to Drive ...')
peft_model.save_pretrained(MERGED_DIR)
try:
    processor = WhisperProcessor.from_pretrained(ready_adapters[first_tier])
except Exception:
    processor = WhisperProcessor.from_pretrained(BASE_MODEL_ID)
processor.save_pretrained(MERGED_DIR)

# 6. Save Metadata
meta = {
    'base_model': BASE_MODEL_ID,
    'adapters_used': tiers_ordered,
    'weights': norm_weights,
    'density': DENSITY,
    'combination': 'ties'
}
with open(MERGED_DIR + '/merge_config.json', 'w') as f:
    json.dump(meta, f, indent=2)

print(f'\n✅ TIES merge finished successfully! Saved → {MERGED_DIR}')
print('   Next step: Open Notebook_6_Evaluate.ipynb to benchmark.')

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CELL 4 — Verify Merged Adapter Output
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
import os
MERGED_DIR = DRIVE_ROOT + '/adapters/merged'
print('Merged adapter directory listing:')
for f in os.listdir(MERGED_DIR):
    sz_kb = os.path.getsize(f'{MERGED_DIR}/{f}') / 1024
    print(f'  {f:<28} ({sz_kb:.1f} KB)')
print('\n✅ Ready for Notebook 6 evaluation!')